In [13]:
# Configuracion
N_ITER = 4
CV = 2
RANDOM_STATE = 42
TARGET_CLASS = None  # None => clase minima de y_train
BASE_NAME = None     # None => tomar ganador del resumen previo
BALANCE_NAME = None  # NONE/SMOTE/UNDER/SMOTEENN


In [14]:
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from lightgbm import LGBMClassifier
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    make_scorer,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import label_binarize

REPO_ROOT = Path('.').resolve()
INPUT_BASE = REPO_ROOT / 'data' / 'intermediate' / '05_seleccion'
PRE_REPORTS_BASE = REPO_ROOT / 'reports' / '09_modelo_lightgbm'
OUTPUT_BASE = REPO_ROOT / 'reports' / '11_tuning_lightgbm'

BALANCE_METHODS = {
    'NONE': None,
    'SMOTE': SMOTE(random_state=RANDOM_STATE),
    'UNDER': RandomUnderSampler(random_state=RANDOM_STATE),
    'SMOTEENN': SMOTEENN(random_state=RANDOM_STATE),
}


def latest_file(pattern_base: Path, pattern: str) -> Path:
    files = sorted(pattern_base.glob(pattern), key=lambda p: p.stat().st_mtime)
    if not files:
        raise FileNotFoundError(f'No se encontro archivo con patron: {pattern_base / pattern}')
    return files[-1]


def latest_input_path() -> Path:
    candidates = sorted([p for p in INPUT_BASE.iterdir() if p.is_dir()])
    if not candidates:
        raise FileNotFoundError(f'No hay subdirectorios en {INPUT_BASE}')
    return candidates[-1]


def choose_candidates(summary_csv: Path) -> list[dict]:
    df = pd.read_csv(summary_csv)
    req = {'modelo', 'balanceo', 'cv_recall_target', 'cv_macro_f1'}
    miss = req - set(df.columns)
    if miss:
        raise ValueError(f'Faltan columnas en resumen: {sorted(miss)}')

    d = df.dropna(subset=['cv_recall_target', 'cv_macro_f1']).copy()
    if d.empty:
        raise ValueError('No hay filas validas con cv_recall_target y cv_macro_f1')

    by_recall = d.sort_values(['cv_recall_target', 'cv_macro_f1'], ascending=False).iloc[0]
    by_general = d.sort_values(['cv_macro_f1', 'cv_recall_target'], ascending=False).iloc[0]

    candidates = [
        {
            'tag': 'best_recall_target',
            'base_name': str(by_recall['modelo']),
            'balance_name': str(by_recall['balanceo']),
            'cv_recall_target': float(by_recall['cv_recall_target']),
            'cv_macro_f1': float(by_recall['cv_macro_f1']),
        },
        {
            'tag': 'best_general',
            'base_name': str(by_general['modelo']),
            'balance_name': str(by_general['balanceo']),
            'cv_recall_target': float(by_general['cv_recall_target']),
            'cv_macro_f1': float(by_general['cv_macro_f1']),
        },
    ]

    dedup = []
    seen = set()
    for c in candidates:
        key = (c['base_name'], c['balance_name'])
        if key in seen:
            continue
        seen.add(key)
        dedup.append(c)
    return dedup


def resolve_target_class(y: pd.Series, target: int | None) -> int:
    classes = list(pd.Series(y).unique())
    if target is None:
        return int(pd.Series(y).min())
    if target in classes:
        return int(target)
    return int(classes[0])


def build_pipeline(balancer, num_classes: int) -> Pipeline:
    model = LGBMClassifier(
        boosting_type='gbdt',
        objective='multiclass' if num_classes > 2 else 'binary',
        n_jobs=1,
        random_state=RANDOM_STATE,
    )
    steps = []
    if balancer is not None:
        steps.append(('balance', clone(balancer)))
    steps.append(('model', model))
    return Pipeline(steps)


def save_eval_pdf(y_true, y_pred, y_proba, class_order, out_pdf: Path, title_prefix: str):
    out_pdf.parent.mkdir(parents=True, exist_ok=True)
    cm = confusion_matrix(y_true, y_pred, labels=class_order)

    with PdfPages(out_pdf) as pdf:
        ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_order).plot(cmap='Blues')
        plt.title(f'{title_prefix} - Matriz de Confusion')
        pdf.savefig(); plt.close()

        if np.unique(y_true).size >= 2:
            y_bin = label_binarize(y_true, classes=class_order)
            proba_for_curves = y_proba

            if y_bin.shape[1] == 1:
                y_bin = np.hstack([1 - y_bin, y_bin])
                if proba_for_curves.shape[1] == 1:
                    proba_for_curves = np.hstack([1 - proba_for_curves, proba_for_curves])

            plt.figure()
            for i, clase in enumerate(class_order):
                fpr, tpr, _ = roc_curve(y_bin[:, i], proba_for_curves[:, i])
                roc_auc = auc(fpr, tpr)
                plt.plot(fpr, tpr, label=f'Clase {clase} (AUC={roc_auc:.2f})')
            plt.plot([0, 1], [0, 1], 'k--')
            plt.title(f'{title_prefix} - Curvas ROC por clase')
            plt.xlabel('FPR')
            plt.ylabel('TPR')
            plt.legend()
            pdf.savefig(); plt.close()

            plt.figure()
            for i, clase in enumerate(class_order):
                precision, recall, _ = precision_recall_curve(y_bin[:, i], proba_for_curves[:, i])
                pr_auc = average_precision_score(y_bin[:, i], proba_for_curves[:, i])
                plt.plot(recall, precision, label=f'Clase {clase} (PR AUC={pr_auc:.2f})')
            plt.title(f'{title_prefix} - Curvas Precision-Recall por clase')
            plt.xlabel('Recall')
            plt.ylabel('Precision')
            plt.legend()
            pdf.savefig(); plt.close()


In [15]:
summary_csv = latest_file(PRE_REPORTS_BASE, '**/resumen_modelos_lightgbm.csv')
input_path = latest_input_path()

if BASE_NAME and BALANCE_NAME:
    candidates = [{
        'tag': 'manual',
        'base_name': BASE_NAME,
        'balance_name': BALANCE_NAME,
        'cv_recall_target': None,
        'cv_macro_f1': None,
    }]
else:
    candidates = choose_candidates(summary_csv)

run_id = datetime.today().strftime('%Y%m%d')
output_dir = OUTPUT_BASE / run_id / f"LightGBM_tuning_{datetime.today().strftime('%Y-%m-%d')}"
output_dir.mkdir(parents=True, exist_ok=True)

all_summary = []

for cand in candidates:
    tag = cand['tag']
    base_name = cand['base_name']
    balance_name = cand['balance_name']

    if balance_name not in BALANCE_METHODS:
        raise ValueError(f'Balanceo no soportado: {balance_name}')

    print(f"\n=== Candidato: {tag} | {base_name} [{balance_name}] ===")

    X_train = pd.read_parquet(input_path / f'X_train_{base_name}.parquet')
    X_test = pd.read_parquet(input_path / f'X_test_{base_name}.parquet')
    X_train = X_train.astype('float32')
    X_test = X_test.astype('float32')
    y_train = pd.read_parquet(input_path / f'y_train_{base_name}.parquet').squeeze()
    y_test = pd.read_parquet(input_path / f'y_test_{base_name}.parquet').squeeze()

    target_class = resolve_target_class(y_train, TARGET_CLASS)
    y_min = int(y_train.min())
    y_train_adj = y_train - y_min
    y_test_adj = y_test - y_min
    target_class_adj = target_class - y_min
    num_classes = int(pd.Series(y_train_adj).nunique())

    balancer = BALANCE_METHODS[balance_name]
    pipeline = build_pipeline(balancer, num_classes)

    scorers = {
        'recall_target': make_scorer(recall_score, labels=[target_class_adj], average='macro', zero_division=0),
        'f1_macro': 'f1_macro',
    }

    param_distributions = {
        'model__n_estimators': [120, 180],
        'model__learning_rate': [0.05, 0.08],
        'model__num_leaves': [31, 63],
        'model__max_depth': [-1, 8],
        'model__subsample': [0.85, 1.0],
        'model__colsample_bytree': [0.85, 1.0],
        'model__min_child_samples': [20, 40],
        'model__reg_alpha': [0.0, 0.1],
        'model__reg_lambda': [1.0, 2.0],
    }

    cv = StratifiedKFold(n_splits=CV, shuffle=True, random_state=RANDOM_STATE)
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=N_ITER,
        scoring=scorers,
        refit='recall_target',
        cv=cv,
        n_jobs=1,
        random_state=RANDOM_STATE,
        verbose=1,
        return_train_score=False,
    )
    search.fit(X_train, y_train_adj)

    best_model = search.best_estimator_
    y_pred_test = best_model.predict(X_test) + y_min
    y_pred_train = best_model.predict(X_train) + y_min
    y_proba = best_model.predict_proba(X_test)
    class_order = best_model.named_steps['model'].classes_ + y_min

    cand_dir = output_dir / tag
    cand_dir.mkdir(parents=True, exist_ok=True)

    pdf_path = cand_dir / 'reporte_tuning_lightgbm.pdf'
    save_eval_pdf(y_test, y_pred_test, y_proba, class_order, pdf_path, f'{tag} - {base_name} [{balance_name}]')

    cv_results = pd.DataFrame(search.cv_results_).sort_values('rank_test_recall_target')
    cv_results.to_csv(cand_dir / 'cv_results_lightgbm_tuning.csv', index=False)

    summary = {
        'candidate_tag': tag,
        'base_name': base_name,
        'balanceo': balance_name,
        'summary_csv_source': str(summary_csv),
        'input_path': str(input_path),
        'target_class_original': int(target_class),
        'selected_cv_recall_target': cand['cv_recall_target'],
        'selected_cv_macro_f1': cand['cv_macro_f1'],
        'cv_best_recall_target': float(search.best_score_),
        'cv_best_f1_macro': float(cv_results.iloc[0]['mean_test_f1_macro']),
        'test_accuracy': float(accuracy_score(y_test, y_pred_test)),
        'test_f1_macro': float(f1_score(y_test, y_pred_test, average='macro', zero_division=0)),
        'train_f1_macro': float(f1_score(y_train, y_pred_train, average='macro', zero_division=0)),
        'pdf_report': str(pdf_path),
    }

    pd.DataFrame([summary]).to_csv(cand_dir / 'resumen_tuning_lightgbm.csv', index=False)
    with open(cand_dir / 'best_params_lightgbm.json', 'w', encoding='utf-8') as f:
        json.dump(search.best_params_, f, indent=2)

    report_test = pd.DataFrame(classification_report(y_test, y_pred_test, output_dict=True, zero_division=0)).T
    report_test.to_csv(cand_dir / 'classification_report_test_lightgbm.csv')

    all_summary.append(summary)
    print(json.dumps(summary, indent=2))

pd.DataFrame(all_summary).to_csv(output_dir / 'resumen_tuning_lightgbm_all_candidates.csv', index=False)
print(f'Salida: {output_dir}')



=== Candidato: best_recall_target | MinMax_ORIGINAL_PCA30 [SMOTEENN] ===
Fitting 2 folds for each of 4 candidates, totalling 8 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.017999 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 98341, number of used features: 30
[LightGBM] [Info] Start training from score -0.773325
[LightGBM] [Info] Start training from score -1.765753
[LightGBM] [Info] Start training from score -2.300564
[LightGBM] [Info] Start training from score -2.445086
[LightGBM] [Info] Start training from score -1.711774
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.037232 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 98402, number of used features: 30
[LightGBM] [Info] S